In [ ]:
import os
import json
import getpass
import google.generativeai as genai
from dotenv import load_dotenv

# .env 파일 로드
dotenv_path = os.path.join("..", ".env")
load_dotenv(dotenv_path)

# 경로 설정
DATA_DIR = os.path.join("..", "data", "proceed")
RAG_DATA_PATH = os.path.join(DATA_DIR, "kf_area_rag_data.json")
OUTPUT_DIR = os.path.join("..", "data", "scenarios")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gemini API Key 세팅
gemini_api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not gemini_api_key:
    gemini_api_key = getpass.getpass("Gemini API Key를 입력하세요: ")
    os.environ["GEMINI_API_KEY"] = gemini_api_key
genai.configure(api_key=gemini_api_key)

print(f"RAG 데이터 경로: {os.path.abspath(RAG_DATA_PATH)}")
print(f"Gemini API 연결 완료.")


✔ RAG 데이터 경로: /Users/ahramkim/Documents/GitHub/k-heroes/backend/data/proceed/kf_area_rag_data.json
✔ Gemini API 연결 완료.


/Users/ahramkim/Documents/GitHub/k-heroes/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/9h/qj2rv5k9005fgsp61dxxrk100000gn/T/ipykernel_4534/1000787852.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. RAG 마스터 데이터 로드

In [ ]:
# RAG 데이터 로드
if os.path.exists(RAG_DATA_PATH):
    with open(RAG_DATA_PATH, "r", encoding="utf-8") as f:
        rag_data = json.load(f)
    print(f"[성공] 총 {len(rag_data)}개의 RAG 단서 데이터를 로드했습니다.")
else:
    print(f"[에러] RAG 데이터 파일을 찾을 수 없습니다. 경로: {RAG_DATA_PATH}")


✔ [성공] 총 24434개의 RAG 단서 데이터를 로드했습니다.


## 2. 인물 검색 및 RAG 역사 컨텍스트 필터링

In [3]:
# 1. 인물명 입력 받기
search_query = input("플레이할 역사 속 인물의 이름을 입력하세요 (예: 이순신, 윤봉길): ").strip()
if not search_query:
    search_query = "이순신"
    print(f"입력값이 없어 기본 인물 '{search_query}'으로 시작합니다.")

# 2. 관련 RAG 문서 필터링
retrieved_clues = []
for item in rag_data:
    metadata = item.get("metadata", {})
    related_person = metadata.get("related_person") or ""
    text = item.get("text") or ""
    
    if search_query in related_person or search_query in text:
        retrieved_clues.append(item)

print(f"\n==================================================")
print(f" 🔍 [{search_query}] 관련 RAG 데이터 검색 결과")
print(f" - 매칭된 단서 수: 총 {len(retrieved_clues)}개")
print(f"==================================================")

# RAG 컨텍스트 문자열 합성
context_str = ""
for i, clue in enumerate(retrieved_clues):
    context_str += f"[단서 {i+1}] {clue['text']}\n\n"

if not retrieved_clues:
    print("[경고] RAG 데이터베이스에 해당 인물 관련 단서가 없습니다. 생성 정확도가 낮아질 수 있습니다.")



 🔍 [이순신] 관련 RAG 데이터 검색 결과
 - 매칭된 단서 수: 총 110개


## 3. Gemini 기반 맞춤형 인물 설명 프로필 카드 생성

In [4]:
# Gemini API 호출하여 인물 설명 카드 생성 (JSON 모드)
character_card_prompt = f"""
너는 역사 선택형 시뮬레이션 게임 'K-Heroes'의 인물 카드 설계자야.
제공된 RAG 백그라운드 지식을 기반으로 대상 인물에 대한 정보를 추출하여 재미있는 인물 카드 데이터로 생성해 줘.
어려운 한자어는 피하고 초등학생도 쉽게 읽을 수 있는 단어를 써줘.

[대상 인물]
{search_query}

[RAG 데이터 컨텍스트]
{context_str}

반드시 아래 JSON 형식으로만 출력해:
{{
    "name": "인물 이름",
    "era": "시대 명칭 (예: 조선 시대 1545~1598)",
    "situation": "당시 시대 상황 설명 (쉽게 2~3줄)",
    "one_line_summary": "히어로물 느낌의 직관적인 수식어 한줄 요약",
    "mbti": "인물 성향에 맞는 MBTI 4글자",
    "mbti_nickname": "MBTI에 따른 캐릭터 별명",
    "mbti_details": {{
        "E_I": "외향/내향형 특성 한줄 설명",
        "S_N": "감각/직관형 특성 한줄 설명",
        "T_F": "사고/감정형 특성 한줄 설명",
        "J_P": "판단/인식형 특성 한줄 설명"
    }},
    "stats": [
        {{"name": "스탯 1 이름 (예: 행동력 🔥)", "stars": "★★★★★", "stars_raw": "5/5", "desc": "왜 그런 스탯을 줬는지 쉬운 이유 설명"}},
        {{"name": "스탯 2 이름 (예: 대담함 🛡️)", "stars": "★★★★★", "stars_raw": "5/5", "desc": "설명"}},
        {{"name": "스탯 3 이름 (예: 위장술 ⚙️)", "stars": "★★★★☆", "stars_raw": "4/5", "desc": "설명"}}
    ],
    "intro_quote": "인물의 유명한 명언이나 다짐 한 줄",
    "intro_desc": "인물이 이 거사를 치르게 된 계기와 시대 상황 설명 2~3줄"
}}
"""

model_json = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

print("인물 카드 데이터를 동적으로 분석 및 생성 중...")
response_card = model_json.generate_content(character_card_prompt)
card_data = json.loads(response_card.text)

# 인물 카드 출력하기
print("\n" + "="*50)
print(f"🦸🏻 K-Heroes 프로필 카드: {card_data['name']}")
print("="*50)
print(f"- 시대: {card_data['era']}")
print(f"- 시대 상황: {card_data['situation']}")
print(f"- 한줄 요약: \"{card_data['one_line_summary']}\"")
print(f"- MBTI: {card_data['mbti']} ({card_data['mbti_nickname']})")
print(f"  • I/E: {card_data['mbti_details']['E_I']}")
print(f"  • N/S: {card_data['mbti_details']['S_N']}")
print(f"  • F/T: {card_data['mbti_details']['T_F']}")
print(f"  • P/J: {card_data['mbti_details']['J_P']}")
print("\n[강점 특기 스탯]")
for stat in card_data['stats']:
    print(f"  • {stat['name']}: {stat['stars']} `[{stat['stars_raw']}]` - {stat['desc']}")
print("\n" + "-"*50)
print(f"📢 \"{card_data['intro_quote']}\"")
print(f"{card_data['intro_desc']}")
print(f"이제 당신이 {card_data['name']} 장군이 되어 중요한 역사적 순간을 플레이합니다. 준비되셨나요?")
print("="*50 + "\n")

# 스탯 목록 및 게임 내 관리용 능력치 사전 초기화
game_stats = {}
for i, stat in enumerate(card_data['stats']):
    game_stats[f"stat_{i+1}"] = {"name": stat['name'], "value": 50}


인물 카드 데이터를 동적으로 분석 및 생성 중...

🦸🏻 K-Heroes 프로필 카드: 이순신
- 시대: 조선 시대 (1545~1598)
- 시대 상황: 일본이 쳐들어와 나라가 큰 위험에 빠졌던 때였어요. 많은 사람이 힘들어했지만, 이순신 장군은 바다에서 용감하게 싸워야 했답니다.
- 한줄 요약: "바다를 지켜낸 불패의 영웅, 거북선의 선장!"
- MBTI: INTJ (전략의 대가, 바다의 설계자)
  • I/E: 내향형 (I): 조용히 깊이 생각하며 혼자만의 시간에 집중하는 것을 좋아했어요.
  • N/S: 직관형 (N): 눈앞의 상황뿐 아니라 멀리 내다보고 큰 그림을 그려내는 데 뛰어났어요.
  • F/T: 사고형 (T): 감정보다는 논리적으로 판단하고 효율적인 방법을 먼저 찾았어요.
  • P/J: 판단형 (J): 목표를 세우면 계획적으로 철저히 실행하고 결정을 미루지 않았어요.

[강점 특기 스탯]
  • 리더십 💪: ★★★★★ `[5/5]` - 부하들과 명나라 장수 진린까지 믿고 따르게 만든 놀라운 리더였어요.
  • 전략 💡: ★★★★★ `[5/5]` - 거센 물살을 이용한 명량 해전, 학익진으로 승리한 한산도 대첩 등 기막힌 작전을 많이 세웠어요.
  • 끈기 🐢: ★★★★★ `[5/5]` - 어려운 상황에서도 포기하지 않고 싸워 마침내 나라를 지켜냈어요.

--------------------------------------------------
📢 "아직 신에게는 12척의 배가 남아있사옵니다!"
일본이 두 번이나 우리나라를 쳐들어왔던 아주 힘든 시기였어요. 이순신 장군은 백성을 지키고 나라를 구하기 위해 거북선이라는 멋진 배를 만들고, 뛰어난 전략으로 바다에서 일본군과 용감하게 싸웠답니다.
이제 당신이 이순신 장군이 되어 중요한 역사적 순간을 플레이합니다. 준비되셨나요?



## 4. 시나리오 플레이 루프 (while문, 최대 STEP 4)

In [5]:
# 게임 상태 값 초기화
history_score = 0
current_step = 1
steps_names = ["발단", "전개", "위기", "절정"]
game_history = []
choices_path = [] # 유저의 선택 루트 (예: ['A', 'B'])

print("🎮 대화형 역사 시뮬레이션 게임 스타트! (조기 역사 승리 조건: 역사점수 100점)")
print("="*80)

while current_step <= 4:
    step_name = steps_names[current_step - 1]
    print(f"\n[STEP {current_step}] {step_name} 단계 진행 중...")
    
    # 1. Gemini를 호출하여 현재 단계의 상황과 선택지 구성
    step_prompt = f"""
    너는 역사 선택형 시뮬레이션 게임 'K-Heroes'의 실시간 시나리오 출제자야.
    
    [대상 인물]
    {card_data['name']}
    
    [RAG 역사 컨텍스트]
    {context_str}
    
    [현재 상태]
    - 단계: STEP {current_step} ({step_name})
    - 과거 선택 이력: {game_history}
    - 현재 능력치 상태: { {k: f"{v['name']}: {v['value']}%" for k, v in game_stats.items()} }
    
    [미션]
    위 상태 정보와 RAG 컨텍스트를 고려하여, K-Heroes에 어울리는 STEP {current_step} 상황을 생성해라.
    1. 실제 역사적 사건(발단-전개-위기-절정 흐름)을 바탕으로 상황을 창작할 것.
    2. 상황(situation)은 모바일 최적화용으로 2~3줄 내외로 짧고 긴박하게 작성할 것. 어려운 한자어는 쓰지 말고 쉬운 한글을 사용할 것.
    3. 선택지 A, B 중 하나는 실제 역사적 사실(True)이어야 하고, 다른 하나는 그럴듯한 가상 대체 역사(Alternative)여야 함.
    4. 토글(Toggle) 설명에는 이 상황과 관련된 숨겨진 역사적 사실이나 보충 정보를 쉽게 2~3줄로 설명할 것.
    5. 선택 결과에 따라 3가지 지표가 어떻게 바뀔지 정의할 것.
    
    반드시 다음 JSON 형식으로만 응답해라:
    {{
        "title": "이번 단계의 짤막한 제목",
        "situation": "처한 메인 상황 2~3줄",
        "toggle_question": "토글 버튼 핵심 질문",
        "toggle_answer": "토글 답변 상세 역사 해설",
        "choice_a": "선택지 A 문장",
        "choice_a_is_historical": true 또는 false,
        "choice_b": "선택지 B 문장",
        "choice_b_is_historical": true 또는 false,
        "stat_effects_a": {{ "stat_1": 10, "stat_2": -5, "stat_3": 15 }},
        "stat_effects_b": {{ "stat_1": -10, "stat_2": 15, "stat_3": -5 }}
    }}
    """
    
    response_step = model_json.generate_content(step_prompt)
    step_data = json.loads(response_step.text)
    
    # 2. 유저에게 메인 상황 및 힌트 출력
    print(f"📌 상황: {step_data['situation']}")
    print(f"💡 [토글 역사 상식] {step_data['toggle_question']}")
    print(f"   => {step_data['toggle_answer']}")
    print(f"\n[유저 선택지]")
    print(f"  A. {step_data['choice_a']}")
    print(f"  B. {step_data['choice_b']}")
    
    # 3. 유저 입력 받기
    user_choice = ""
    while user_choice not in ["A", "B"]:
        user_choice = input("선택하세요 (A 또는 B 입력): ").strip().upper()
    
    # 4. 결과 처리
    choices_path.append(user_choice)
    chosen_is_historical = False
    effects = None
    chosen_text = ""
    
    # Handle potentially string-typed boolean variables from API output
    is_historical_a = step_data.get("choice_a_is_historical")
    if isinstance(is_historical_a, str):
        is_historical_a = (is_historical_a.lower() == "true")
        
    is_historical_b = step_data.get("choice_b_is_historical")
    if isinstance(is_historical_b, str):
        is_historical_b = (is_historical_b.lower() == "true")
        
    if user_choice == "A":
        chosen_is_historical = is_historical_a
        effects = step_data["stat_effects_a"]
        chosen_text = step_data["choice_a"]
    else:
        chosen_is_historical = is_historical_b
        effects = step_data["stat_effects_b"]
        chosen_text = step_data["choice_b"]
        
    # 역사적 결정 여부에 따라 역사 점수 50점 증가
    if chosen_is_historical:
        history_score += 50
        print("\n🟢 [선택 결과] 실제 역사적 사실과 일치하는 옳은 선택을 내렸습니다!")
        print(f" 📈 역사 점수 +50점 (현재 역사 점수: {history_score}/100점)")
    else:
        print("\n🔵 [선택 결과] 대체 역사 시나리오 분기로 진입했습니다. (실제 역사와는 다른 결정)")
        print(f" 📉 역사 점수 변동 없음 (현재 역사 점수: {history_score}/100점)")
        
    # 스탯 지표 값 업데이트
    for key in ["stat_1", "stat_2", "stat_3"]:
        val = effects.get(key, 0)
        game_stats[key]["value"] = max(0, min(100, game_stats[key]["value"] + val))
        sign = "+" if val >= 0 else ""
        print(f"  🛡️ {game_stats[key]['name']}: {game_stats[key]['value']}% ({sign}{val}%)")
        
    # 진행 내역 기록
    game_history.append({
        "step": current_step,
        "situation": step_data["situation"],
        "user_choice": user_choice,
        "chosen_text": chosen_text,
        "is_historical": chosen_is_historical
    })
    
    # 5. 조기 종료 조건 검증 (역사 100점 달성 시)
    if history_score >= 100:
        print("\n🏆========================================================================🏆")
        print(" 🎉 대단합니다! 역사와 100% 일치하는 선택을 연달아 성공하여 역사 점수 100점을 달성했습니다!")
        print("    조국을 구한 위대한 행적을 그대로 밟아 거사에 조기 대성공하였습니다.")
        print("🏆========================================================================🏆")
        break
        
    current_step += 1

print("\n" + "="*80)
print("🏁 게임 시뮬레이션 종료! 최종 엔딩을 도출하고 있습니다...")
print("="*80)


🎮 대화형 역사 시뮬레이션 게임 스타트! (조기 역사 승리 조건: 역사점수 100점)

[STEP 1] 발단 단계 진행 중...
📌 상황: 해미읍성의 수비 책임자가 된 당신. 왜구의 침입 소문이 심상치 않다. 약탈과 노략질에 대비하여 해안 방비를 강화해야 할 시급한 상황이다.
💡 [토글 역사 상식] 해미읍성과 이순신 장군은 어떤 인연이 있었을까요?
   => 해미읍성은 이순신 장군이 젊은 시절 충청도 병마절도사(병사)의 군관으로 근무하며 해안 방어에 힘썼던 곳입니다. 당시 왜구의 침입이 잦아 해안 방비의 중요성이 매우 컸습니다.

[유저 선택지]
  A. 병사들을 직접 훈련시키고, 해미읍성의 방어 체계를 꼼꼼히 점검하며 만일의 사태에 대비한다.
  B. 주변 마을에 급히 전령을 보내 더 많은 백성을 징발하여 성벽을 높이는 데 집중한다.

🟢 [선택 결과] 실제 역사적 사실과 일치하는 옳은 선택을 내렸습니다!
 📈 역사 점수 +50점 (현재 역사 점수: 50/100점)
  🛡️ 리더십 💪: 65% (+15%)
  🛡️ 전략 💡: 60% (+10%)
  🛡️ 끈기 🐢: 55% (+5%)

[STEP 2] 전개 단계 진행 중...
📌 상황: 해미읍성 방비를 성공적으로 마친 당신에게 의외의 명령이 떨어진다. 북방 오랑캐의 침입이 잦아지자, 조정은 당신을 북쪽 국경 지대로 파견하려 한다.
💡 [토글 역사 상식] 이순신은 왜 북방으로 가게 되었을까요?
   => 이순신 장군은 임진왜란 발발 전 여러 차례 북방 국경 지대에서 여진족을 방어하며 실전 경험을 쌓았습니다. 이는 훗날 임진왜란에서 수군을 이끄는 데 큰 밑거름이 됩니다.

[유저 선택지]
  A. 새로운 전장에서도 최선을 다하겠다는 각오로 북방으로 향한다.
  B. 해미읍성의 안정된 방비를 이유로 북방 파견을 거절하고 이곳에 남기를 청한다.

🔵 [선택 결과] 대체 역사 시나리오 분기로 진입했습니다. (실제 역사와는 다른 결정)
 📉 역사 점수 변동 없음 (현재 역사 점수: 50/100점)
  🛡️ 리더십 💪: 7

## 5. 최종 시뮬레이션 엔딩 결과 생성 및 파일 저장

In [ ]:
# 최종 엔딩 카드 생성 (유저의 히스토리와 스탯 적용)
ending_prompt = f"""
너는 역사 시뮬레이션 게임 'K-Heroes'의 최종 엔딩 연출가야.
유저가 게임 도중 내린 선택 이력과 최종 능력치 스탯을 토대로 최종 감동적인 엔딩을 작성해 줘.

[대상 인물]
{card_data['name']}

[RAG 역사 컨텍스트]
{context_str}

[유저의 플레이 역사]
- 진행 단계: {len(game_history)}단계까지 플레이함 (역사 점수: {history_score}/100점)
- 유저의 선택 이력: {[f'STEP {item["step"]}: {item["chosen_text"]} (역사적 여부: {item["is_historical"]})' for item in game_history]}
- 최종 지표 능력치: { {v['name']: f"{v['value']}%" for v in game_stats.values()} }

# 제작 가이드라인 (최종 결과)
1. 엔딩 타이틀:
   - 만약 역사 점수가 100점(조기 성공 포함)인 경우: 실제 역사와 100% 일치하는 찬란한 승리(True Ending) 타이틀을 달아주세요. (🔴 빨간색 동그라미 기재)
   - 역사 점수가 100점 미만인 경우: 유저의 선택에 따른 개연성 있는 가상 시나리오 엔딩(Alternative Ending) 타이틀을 달아주세요. (🔵 파란색 동그라미 기재)
2. 실제 역사적 사실과 비교:
   - 실제 역사와 어떤 점이 다르고 어떤 역사적 교훈이 있는지 팩트체크 형식으로 기술해 주세요.
3. 결과 스토리 및 요약:
   - '📖 내가 만든 이야기 (Story)' 부분은 효과음이나 대사로 시작하는 강렬한 헤드라인 1줄과 결과 서사 2줄로 짧게 작성해 주세요.
   - '📌 결과 요약 (Summary)'은 쉬운 단어의 불릿 포인트 3개로 요약해 주세요.
4. 초등학생 눈높이에 맞게 쉽고 웅장한 히어로물 톤을 유지하세요.

# 출력 포맷
### 3. 최종 결과

#### RESULT. [유저의 선택조합: {'-'.join(choices_path)}]
**[상단 타이틀 및 캐릭터 연출]**
- **엔딩 타이틀:** 엔딩 제목 (실제 역사와 100% 일치 또는 가상 시나리오 표기)
- **최종 능력치 결과:** (3가지 능력치의 최종 수치 표기)

**[실제 역사적 사실과 비교]**
- **💡 이 엔딩은 실제 역사와 어떤 차이가 있을까요?**
- (팩트체크와 역사적 의의 서술)

**[결과 스토리 및 요약]**
- **📖 내가 만든 이야기 (Story)**
  - **\"[효과음이나 대사 중심의 강렬한 헤드라인]\"**
  - (결과 스토리 2줄)
- **📌 결과 요약 (Summary)**
  - (결과 요약 불릿 포인트 3개)

#### 추천 방문지
- 💡 {card_data['name']}을(를) 좀 더 알아보고 싶으세요?
- (생가, 기념관, 박물관 등 추천 2곳)

#### Floating Button
- **공유하기:** \"내가 만든 역사 엔딩을 친구에게 공유해보세요! 🔗\"
- **다음 인물 체험하기**
"""

model_text = genai.GenerativeModel("gemini-2.5-flash")
response_ending = model_text.generate_content(ending_prompt)
ending_result = response_ending.text

# 최종 결과 마크다운 화면 출력
print("\n" + "="*80)
print(ending_result)
print("="*80 + "\n")

# 최종 결과 저장
safe_name = card_data['name'].replace(" ", "").replace("/", "_")
output_file_name = f"04_{safe_name}_interactive_simulation.md"
output_file_path = os.path.join(OUTPUT_DIR, output_file_name)

with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(ending_result)

print(f"✔ 게임 결과 리포트 저장 완료: {os.path.abspath(output_file_path)}")



### 3. 최종 결과

#### RESULT. [유저의 선택조합: A-B-A]
**[상단 타이틀 및 캐릭터 연출]**
- **엔딩 타이틀:** 실제 역사와 100% 일치하는 찬란한 승리(True Ending) 🔴
- **최종 능력치 결과:** 리더십 💪 85%, 전략 💡 60%, 끈기 🐢 55%

**[실제 역사적 사실과 비교]**
- **💡 이 엔딩은 실제 역사와 어떤 차이가 있을까요?**
  - 당신은 이순신 장군이 해미읍성에서 병사들을 직접 훈련시키고 방어 체계를 꼼꼼히 점검하며 만일의 사태에 대비한 첫걸음(STEP 1)을 선택했습니다. 이는 실제 이순신 장군이 해미읍성(단서 23, 34, 53, 60)에서 병마절도사로 재직하며 군비 확충에 힘썼던 역사적 사실과 일치합니다.
  - 하지만 두 번째 선택(STEP 2)에서 해미읍성의 안정된 방비를 이유로 북방 파견을 거절하고 남기를 청하는 가상의 선택을 했습니다. 실제 역사에서 이순신 장군은 북방 오랑캐 방비에 활약(단서 1)하기도 했습니다. 비록 게임 속 이 선택은 실제 역사와 다르지만, 당신이 보여준 확고한 방어 의지와 뛰어난 리더십(85%)은 장군이 어떤 위치에 있든 나라를 지키는 데 최선을 다했으리라는 상상을 가능하게 합니다.
  - 마지막으로 소수의 정예 병력을 이끌고 직접 해안 정찰에 나서, 적의 규모와 의도를 파악하고 선제적으로 대비한 선택(STEP 3)은 장군의 뛰어난 전략적 통찰력(60%)과 용맹함, 그리고 현명한 준비 정신을 보여줍니다. 이는 명량(단서 2, 77), 한산(단서 4, 79) 등 수많은 해전에서 기적 같은 승리를 이끌어낸 이순신 장군의 실제 모습과 완벽하게 일치합니다.

**[결과 스토리 및 요약]**
- **📖 내가 만든 이야기 (Story)**
  - **"두려움을 모르는 영웅의 기개여, 바다를 가르고 나라를 구하리라!"**
  - 당신의 이순신은 흔들림 없는 리더십과 치밀한 전략으로 조선의 바다를 수호하며, 임진왜란의 거친 파도 속에서 빛나는 희망이 되어 찬란한 승리